# LEDGAR (Task 3) — Dataset Exploration

**LexGLUE / LEDGAR** — provision-type classification. Given a single contract provision, classify **what type of provision it is** (e.g. *Governing Laws*, *Terminations*, *Amendments*) by choosing **one label out of a fixed list of 100**.

This is the Task 3 counterpart of `CUAD_dataset_exploration.ipynb`. Where T1 asked a Yes/No question and T2 asked for a JSON value, T3 is plain **multi-class classification** — so this EDA is about the label space, not context/answer column pairs.

We use the research-standard **LexGLUE LEDGAR subset** (`coastalcph/lex_glue`, config `ledgar`), downloaded locally by `scripts/download_ledgar.py`:

- ~80,000 provisions (60k train / 10k validation / 10k test)
- the **100 most frequent labels**, exactly **one label per provision**
- a ready-made split (by SEC filing year) — no contract-leakage problem to solve manually

**Four questions to answer before any training** (see `docs/task_3/TASK3_PLAN.md`, Step 2):
1. **Label balance** — how many examples per label? Decides whether macro-F1 or accuracy is the honest headline metric.
2. **Text length** — do provisions fit the `max_length=1024` training window?
3. **Prompt length** — the instruction must list all 100 labels (~300–400 tokens); confirm instruction + longest provision still fits 1024.
4. **Eyeball examples** — sanity-check that the labels make sense.

In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import dotenv

dotenv.load_dotenv()
pd.set_option("display.max_colwidth", 120)

## Load the LEDGAR splits

`scripts/download_ledgar.py` writes three plain CSVs (`text, label, label_name`) plus a `labels.json` id→name map to `data/LEDGAR/` (git-ignored, like CUAD). Everything downstream reads those flat files.

In [ ]:
DATA_DIR = os.getenv("DATA_DIR", "data")
# LEDGAR lives in its own folder, sibling to CUAD_v1.
LEDGAR_DIR = os.getenv("LEDGAR_DIR", os.path.join(DATA_DIR, "LEDGAR"))
print(f"Using LEDGAR_DIR: {LEDGAR_DIR}")

# Human-readable label map produced once at download time.
with open(os.path.join(LEDGAR_DIR, "labels.json"), encoding="utf-8") as fh:
    id2label = {int(k): v for k, v in json.load(fh).items()}
LABELS = [id2label[i] for i in range(len(id2label))]
print(f"{len(LABELS)} labels")
print(f"  first 5: {LABELS[:5]}")
print(f"  last  5: {LABELS[-5:]}")

splits = {}
for split in ["train", "validation", "test"]:
    df = pd.read_csv(os.path.join(LEDGAR_DIR, f"ledgar_{split}.csv"))
    splits[split] = df
    print(f"{split:>10}: {len(df):>6} rows | columns={list(df.columns)}")

train_df, val_df, test_df = splits["train"], splits["validation"], splits["test"]
train_df.head(3)

## Question 1 — Label balance

How many examples per label? LEDGAR is imbalanced (some provision types are much more common than others). If a handful of labels dominate, plain accuracy can look high while rare classes fail completely — so **macro-F1** (which averages F1 over the 100 labels equally) becomes the honest headline metric, exactly as with T1. We also confirm every one of the 100 labels appears in **all three splits** (needed for the training sanity checks later).

In [ ]:
counts = train_df["label_name"].value_counts()

print(f"Labels present in train : {counts.shape[0]}/100")
print(f"Most common label       : {counts.idxmax()!r} = {counts.max()}")
print(f"Least common label      : {counts.idxmin()!r} = {counts.min()}")
print(f"Imbalance ratio (max/min): {counts.max() / counts.min():.1f}x")
print(f"Mean / median per label : {counts.mean():.0f} / {counts.median():.0f}")

print("\nPer-split label coverage:")
for name, df in splits.items():
    present = df["label_name"].unique()
    missing = sorted(set(LABELS) - set(present))
    flag = f" | MISSING: {missing}" if missing else ""
    print(f"  {name:>10}: {len(present)}/100 labels present{flag}")

print("\nTop 10 labels:")
print(counts.head(10).to_string())
print("\nBottom 10 labels:")
print(counts.tail(10).to_string())

plt.figure(figsize=(15, 6))
counts.plot(kind="bar")
plt.title("LEDGAR train — examples per label (100 labels, sorted)")
plt.ylabel("Count")
plt.xticks(fontsize=6, rotation=90)
plt.tight_layout()
plt.show()

# Insight: a large max/min ratio => strong imbalance => report macro-F1 as the
# headline (rare provision types count as much as common ones), micro-F1 for
# comparability with the LexGLUE leaderboard.

## Question 2 — Provision length (in tokens)

Training truncates at `max_length=1024`. LEDGAR provisions are short (usually well under 200 tokens), but we verify with the **actual Llama-3.1 tokenizer** (the one the fine-tune uses) so the numbers are exact rather than a word-count guess. If the gated tokenizer can't be loaded we fall back to a public proxy so the notebook still runs on any machine.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B"  # same base model the T3 fine-tune uses
MAX_LENGTH = 1024                              # training truncation length

def get_tokenizer():
    try:
        tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.getenv("HF_TOKEN"))
        print(f"Using tokenizer: {MODEL_NAME}")
        return tok, "llama-3.1"
    except Exception as e:
        print(f"Could not load {MODEL_NAME} ({type(e).__name__}: {e}).")
        print("Falling back to the gpt2 tokenizer as a rough proxy.")
        return AutoTokenizer.from_pretrained("gpt2"), "gpt2-proxy"

tokenizer, tok_kind = get_tokenizer()

def token_len(text):
    return len(tokenizer(str(text), add_special_tokens=False)["input_ids"])

def token_lengths(texts, batch_size=1000):
    texts = [str(t) for t in texts]
    lengths = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], add_special_tokens=False)
        lengths.extend(len(ids) for ids in enc["input_ids"])
    return np.array(lengths)

In [ ]:
# Measure over train + validation (the splits we actually fine-tune / eval on).
train_val_text = pd.concat([train_df, val_df], ignore_index=True)["text"].tolist()
prov_lengths = token_lengths(train_val_text)

print(f"Provision length in tokens over {len(prov_lengths)} train+val provisions ({tok_kind}):")
for p in [50, 90, 95, 99]:
    print(f"  p{p:<3}: {int(np.percentile(prov_lengths, p)):>5}")
print(f"  mean: {prov_lengths.mean():>5.1f}")
print(f"  max : {int(prov_lengths.max()):>5}")
print(f"  provisions alone exceeding {MAX_LENGTH}: {(prov_lengths > MAX_LENGTH).sum()}")

plt.figure(figsize=(11, 5))
plt.hist(prov_lengths, bins=60)
plt.axvline(MAX_LENGTH, color="red", linestyle="--", label=f"max_length = {MAX_LENGTH}")
plt.title("LEDGAR provision length (tokens, train+val)")
plt.xlabel("tokens")
plt.ylabel("provisions")
plt.legend()
plt.tight_layout()
plt.show()

## Question 3 — Full-prompt length with the 100-label instruction

The model can't pick a label it hasn't been shown, so the **instruction must contain the full list of 100 allowed labels** (this is also what makes the baseline comparison fair). That list alone is ~300–400 tokens. Here we build the exact T3 prompt — same `### Instruction / ### Input / ### Response` template as T1/T2 — and confirm that **instruction + template + the longest provision** still fits inside 1024 tokens.

In [ ]:
# T3 instruction — the completion is exactly one label name from this list.
LABEL_LIST_STR = ", ".join(LABELS)
INSTRUCTION = (
    "Classify the following contract provision. "
    f"Answer with exactly one label from this list: [{LABEL_LIST_STR}]."
)
# Same prompt template as T1/T2 (completion = the label string).
PROMPT_TEMPLATE = "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n"

instr_tokens = token_len(INSTRUCTION)
template_overhead = token_len(PROMPT_TEMPLATE.format(instruction="", input=""))
longest = int(prov_lengths.max())
worst_case = instr_tokens + template_overhead + longest

print(f"Instruction (with all 100 labels): {instr_tokens:>5} tokens")
print(f"Prompt template overhead         : {template_overhead:>5} tokens")
print(f"Longest provision                : {longest:>5} tokens")
print(f"Worst-case full prompt           : ~{worst_case} tokens  (limit {MAX_LENGTH})")
print("  => FITS" if worst_case < MAX_LENGTH else "  => DOES NOT FIT — longest provisions get truncated")

# Exact full-prompt length distribution (every provision + fixed instruction/template).
full_lengths = prov_lengths + instr_tokens + template_overhead
over = int((full_lengths > MAX_LENGTH).sum())
print(f"\nFull-prompt length: p95={int(np.percentile(full_lengths, 95))} "
      f"p99={int(np.percentile(full_lengths, 99))} max={int(full_lengths.max())}")
print(f"Full prompts exceeding {MAX_LENGTH}: {over} ({100 * over / len(full_lengths):.2f}%)")

print("\n--- Instruction preview (first 400 chars) ---")
print(INSTRUCTION[:400] + " ...")
print("\n--- Example full prompt ---")
print(PROMPT_TEMPLATE.format(instruction=INSTRUCTION, input=train_df["text"].iloc[0]) + train_df["label_name"].iloc[0])

## Question 4 — Eyeball examples per label

Print a few provisions for the most common and the rarest labels to sanity-check that the labels are sensible and the text is clean. This also surfaces plausibly-confusable label pairs (e.g. *Governing Laws* vs *Jurisdictions*) to watch for in the eval's top-confusions report.

In [ ]:
# 3 most common + 2 rarest labels, a couple of provisions each.
sample_labels = list(counts.index[:3]) + list(counts.index[-2:])

for lab in sample_labels:
    subset = train_df[train_df["label_name"] == lab]["text"]
    print("=" * 80)
    print(f"LABEL: {lab!r}   (train count: {int(counts[lab])})")
    for i, text in enumerate(subset.head(2).tolist(), 1):
        print(f"  [{i}] {str(text)[:320]}")
print("=" * 80)

## Takeaways for Task 3

What this EDA actually found (numbers from the run above):

- **Metric choice (Q1):** LEDGAR is heavily imbalanced — **137×** between the most common label (*Governing Laws*, 3167) and the rarest (*Books*, 23). So **macro-F1 is the headline** (rare provision types count as much as common ones) and **micro-F1** is reported for comparability with the LexGLUE leaderboard (encoder models score ~87–88 micro / ~82 macro).
- **⚠️ Not every label is in every split:** all 100 labels appear in **train** and **test**, but **validation is missing `Books`** (99/100). The plan's Step-3 sanity check ("every label appears in both splits") therefore **cannot be strictly met for validation** — either relax it to *train* coverage, or accept that the rarest few labels (`Books` has only 23 train / 0 val examples) won't be evaluable on the validation split. Worth deciding before stratified sampling.
- **Sequence budget (Q2 + Q3):** provisions are short (p50 = 104, p95 = 379 tokens), and the 100-label instruction is **331 tokens** + 9 template tokens. For the vast majority of examples the full prompt fits well under `max_length=1024` (full-prompt p95 = 719, p99 = 925). **But the tail is not zero:** the longest provision is 1749 tokens, and **~0.47% of full prompts (327) exceed 1024** and will be right-truncated. That's an acceptable, small loss — but the instruction eats ~330 tokens of every prompt, so keep an eye on it if `max_length` is ever reduced.
- **Next step (Step 3):** stratified-subsample the 60k train / 10k val down to ~60–100 examples per label (train) and ~10–20 per label (val), then emit `ledgar_task3_train.jsonl` / `ledgar_task3_validation.jsonl` in the shared T1/T2 schema, reusing the exact `INSTRUCTION` and `PROMPT_TEMPLATE` built in Q3. Note the imbalance: rare labels (`Books`, `Assigns`, `Qualifications`) have fewer train examples than the per-label target, so they cap out below the sample size.